- Actor (策略 $\pi_\theta$): 负责动作输出
- Critic (价值 $V_\phi$): 负责给动作（状态）打分
- AC 的核心在于引入了 Advantage (优势函数) 来替代原始的回报 $G_t$：
    - $A(s, a) = Q(s, a) - V(s)$
        - $V(s) = \sum_{a} \pi(a|s) \cdot Q(s, a)$
        - $Q(s, a) \approx r + \gamma V(s')$
        - $A(s, a) = Q(s, a) - V(s)\approx r + \gamma V(s') - V(s)$ (TD error)

### 为什么需要一个不准的 critic

> critic 不需要完美预测真实价值；它只需要把极其嘈杂的回报信号变成更稳定、更局部、更可学习的 advantage 信号。
- “不准的 critic”仍然有用的原因之一：它只要能提供一个大致 baseline，就能显著降低方差。
- 对于 baseline 型 critic，只要 $V(s_t)$ 不依赖当前动作 $a_t$，即使估计偏差很大，policy gradient 的期望方向仍然可以保持无偏；差的主要是方差大小。数学上：
    - $\mathbb{E}_{a_t\sim \pi}[\nabla_\theta \log \pi_\theta(a_t\mid s_t) b(s_t)]=b(s_t)\nabla_\theta \sum_a \pi_\theta(a\mid s_t)=b(s_t)\nabla_\theta 1=0$
    - 所以减去一个状态相关 baseline，不会系统性改变 actor 的期望更新方向。它像秤上的“去皮”功能：皮重估得不精确，读数仍可能更稳定。
- 第二个作用是 bootstrap。很多 RL 任务不能等完整 episode 结束才学习。critic 可以用 TD 形式做局部估计：
    - $A_t \approx r_t + \gamma V(s_{t+1}) - V(s_t)$
- 第三个作用是 credit assignment，也就是把最终结果拆给中间动作。尤其在 LLM agent、RLHF、RLVR 里，reward 往往是 sparse 的：整段回答正确/错误，任务成功/失败，代码测试通过/失败。critic 可以估计每个 token、每个 step、每个状态的相对价值，从而让 actor 知道哪些局部决策更值得加强。
- GAE of PPO
    - $A_t^{\text{GAE}}=\sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l}$
        - $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$
    - λ 越接近 1，越像 Monte Carlo，bias 小、variance 大；λ 越接近 0，越像 TD，bias 大、variance 小。

### 方差（Variance）与偏差（Bias）tradeoff

$$
A(s, a) = Q(s, a) - V(s)\approx r + \gamma V(s') - V(s)
$$
$$
A(s, a) \approx \underbrace{r + \gamma V(s_{t+1})}_{\text{Target (Bootstrap)}} - \underbrace{V(s_t)}_{\text{Baseline}}
$$
- Critic 的存在是为了引入偏差（Bias）来大幅降低方差（Variance），从而加速 Actor 的收敛。
    - bias: $V(s_{t+1})$ (Bootstrap), 使用 $r + \gamma V(s_{t+1})$ 来代替真实回报 $G_t$，这才是引入 Bias 的来源
        - $V(s_{t+1})$ 用来截断后续轨迹的。
    - variance：通过“截断”未来。 你不再依赖未来几千步的随机结果，而是用 Critic 的预测值直接替代了未来。
    - $-V(s_t)$: 减去一个只与 State 有关的基线，不会引入偏差（数学上梯度的期望不变），但能进一步降低方差。